# P1: Framing a Data Problem

**Name:** Ali Gulzar  
**Unity ID:** agulzar2
**Course:** DSA 405-002, Fall 2026  
**Project:** Temperature near Cushing and WTI crude-oil price changes

## 1. The Question

**Research question:** Is there an association between daily average temperature near Cushing, Oklahoma, and daily percentage changes in the Cushing WTI crude-oil spot price from 2015 through 2025?

### Motivation

Weather data may contain information relevant to physical energy markets because temperature can affect energy demand and operating conditions related to petroleum transportation and storage. Cushing is closely tied to the WTI crude-oil benchmark, so this project will test whether temperature near this market hub is associated with short-term WTI price movements. If an association exists, temperature may be useful as one input in a broader oil-market model. If the association is weak or absent, the result would show that local temperature by itself offers little information about daily WTI movements.

The wording intentionally asks about **association rather than causation**. Daily average temperature will be calculated as `(TMAX + TMIN) / 2`, and daily WTI percentage change will be calculated from consecutive trading-day prices. The eventual analysis table will have one row per matched date, with temperature and WTI price-change columns.

## 2. Sources and Access Evidence

### Source 1: EIA daily WTI spot price

| Item | Description |
|---|---|
| **Publisher** | U.S. Energy Information Administration (EIA) |
| **Dataset** | Cushing, OK WTI Spot Price FOB (series `RWTC`) |
| **Dataset page** | [EIA: Cushing, OK WTI Spot Price FOB—Daily](https://www.eia.gov/dnav/pet/hist/RWTCD.htm) |
| **API endpoint** | [EIA Open Data API v2—petroleum spot prices](https://api.eia.gov/v2/petroleum/pri/spt/data/) |
| **Coverage** | Daily WTI spot price in U.S. dollars per barrel; the EIA series begins in 1986 |
| **Project time span** | January 1, 2015 through December 31, 2025 |
| **Approximate project size** | 2,760 trading-day observations, verified below |
| **Access method** | Documented JSON API using the public `DEMO_KEY`; no login or paywall |
| **Variables used** | `period` (date) and `value` (dollars per barrel) |

The following code makes one request, verifies the HTTP status, converts the JSON response to a DataFrame, and displays a row count and preview.
(Note that AI was used to assist with the code below, only to prove accessibility of the source)

In [ ]:
import io
import json
import urllib.parse
import urllib.request
import pandas as pd

HEADERS = {
    "User-Agent": "DSA405 student project (educational data access)"
}

EIA_API_URL = "https://api.eia.gov/v2/petroleum/pri/spt/data/"
eia_params = [
    ("api_key", "DEMO_KEY"),
    ("frequency", "daily"),
    ("data[0]", "value"),
    ("facets[series][]", "RWTC"),
    ("start", "2015-01-01"),
    ("end", "2025-12-31"),
    ("sort[0][column]", "period"),
    ("sort[0][direction]", "asc"),
    ("offset", "0"),
    ("length", "5000"),
]

eia_request_url = EIA_API_URL + "?" + urllib.parse.urlencode(eia_params)
eia_request = urllib.request.Request(eia_request_url, headers=HEADERS)

with urllib.request.urlopen(eia_request, timeout=120) as response:
    eia_status = response.status
    eia_payload = json.load(response)

oil = (
    pd.DataFrame(eia_payload["response"]["data"])[["period", "value"]]
    .rename(columns={"period": "date", "value": "wti_price_usd_per_barrel"})
)
oil["date"] = pd.to_datetime(oil["date"])
oil["wti_price_usd_per_barrel"] = pd.to_numeric(
    oil["wti_price_usd_per_barrel"]
)
oil = oil.sort_values("date").reset_index(drop=True)
oil["daily_return_pct"] = (
    oil["wti_price_usd_per_barrel"].pct_change(fill_method=None) * 100
)

print(f"EIA HTTP status: {eia_status}")
print(f"EIA rows returned (2015-2025): {len(oil):,}")
print(oil.head().to_string(index=False))

EIA HTTP status: 200
EIA rows returned (2015-2025): 2,760
      date  wti_price_usd_per_barrel  daily_return_pct
2015-01-02                     52.72               NaN
2015-01-05                     50.05         -5.064492
2015-01-06                     47.98         -4.135864
2015-01-07                     48.69          1.479783
2015-01-08                     48.80          0.225919


### Source 2: NOAA/NCEI daily temperature

The NOAA station located directly in Cushing (`USC00342318`) stopped reporting daily observations before this project's 2015 start date. Therefore, I selected **Stillwater 2 W** (`USW00053926`), a NOAA station approximately 20.3 miles (32.7 km) from central Cushing that covers the full study period.

| Item | Description |
|---|---|
| **Publisher** | NOAA National Centers for Environmental Information (NCEI) |
| **Dataset** | Global Historical Climatology Network–Daily (GHCN-Daily), Version 3; station `USW00053926`, Stillwater 2 W |
| **Dataset documentation** | [NCEI GHCN-Daily metadata and access page](https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00861) |
| **Actual data request** | [NCEI Daily Summaries web service—Stillwater 2 W, 2015–2025](https://www.ncei.noaa.gov/access/services/data/v1?dataset=daily-summaries&stations=USW00053926&startDate=2015-01-01&endDate=2025-12-31&format=csv&units=standard&includeAttributes=false) |
| **Coverage** | Daily station weather observations; this project uses daily maximum and minimum air temperature in degrees Fahrenheit |
| **Project time span** | January 1, 2015 through December 31, 2025 |
| **Approximate project size** | 4,013 daily rows; 3,991 rows have both TMAX and TMIN, verified below |
| **Access method** | Documented NCEI Daily Summaries web service returning CSV; no key, login, or paywall |
| **Variables used** | `DATE`, `TMAX`, and `TMIN`; average temperature is derived as `(TMAX + TMIN) / 2` |

The following code makes one request, verifies the HTTP status, reads only the needed fields, and displays a row count and preview.

In [ ]:
NOAA_URL = (
    "https://www.ncei.noaa.gov/access/services/data/v1"
    "?dataset=daily-summaries"
    "&stations=USW00053926"
    "&startDate=2015-01-01"
    "&endDate=2025-12-31"
    "&format=csv"
    "&units=standard"
    "&includeAttributes=false"
)

noaa_request = urllib.request.Request(NOAA_URL, headers=HEADERS)
with urllib.request.urlopen(noaa_request, timeout=120) as response:
    noaa_status = response.status
    noaa_csv = response.read().decode("utf-8")

weather_raw = pd.read_csv(
    io.StringIO(noaa_csv),
    usecols=["STATION", "DATE", "TMAX", "TMIN"],
)
weather = weather_raw.rename(
    columns={"DATE": "date", "TMAX": "temp_max_f", "TMIN": "temp_min_f"}
).copy()
weather["date"] = pd.to_datetime(weather["date"])
weather["temp_avg_f"] = (
    weather["temp_max_f"] + weather["temp_min_f"]
) / 2

complete_temperature_rows = weather[["temp_max_f", "temp_min_f"]].notna().all(axis=1).sum()

print(f"NOAA HTTP status: {noaa_status}")
print(f"NOAA rows returned (2015-2025): {len(weather):,}")
print(f"Rows with both TMAX and TMIN: {complete_temperature_rows:,}")
print(weather.head().to_string(index=False))

NOAA HTTP status: 200
NOAA rows returned (2015-2025): 4,013
Rows with both TMAX and TMIN: 3,991
    STATION       date  temp_max_f  temp_min_f  temp_avg_f
USW00053926 2015-01-01        31.0        23.0        27.0
USW00053926 2015-01-02        36.0        31.0        33.5
USW00053926 2015-01-03        41.0        29.0        35.0
USW00053926 2015-01-04        29.0        16.0        22.5
USW00053926 2015-01-05        43.0        14.0        28.5


### How the sources will be combined

The EIA and NOAA datasets will be joined using the calendar date. Because oil prices are reported on trading days while weather is reported daily, an **inner join** will keep dates present in both sources. Rows missing either the daily WTI return or the derived average temperature will be removed before analysis.

The resulting one-to-one analysis table contains:

| Column | Meaning |
|---|---|
| `date` | Matched EIA/NOAA calendar date |
| `wti_price_usd_per_barrel` | EIA WTI spot price |
| `daily_return_pct` | Percentage change from the previous WTI trading-day observation |
| `temp_min_f` | NOAA daily minimum temperature |
| `temp_max_f` | NOAA daily maximum temperature |
| `temp_avg_f` | Derived daily average temperature |

The join below is included to demonstrate that the proposed combined table can actually be constructed. P1 stops at validating the plan; later milestones can use a scatterplot, Pearson correlation, and a regression while accounting for seasonality and other possible confounders.

In [ ]:
combined = (
    oil.merge(
        weather[["date", "temp_min_f", "temp_max_f", "temp_avg_f"]],
        on="date",
        how="inner",
    )
    .dropna(subset=["daily_return_pct", "temp_avg_f"])
    .reset_index(drop=True)
)

print(f"Usable joined rows: {len(combined):,}")
print(f"Date range: {combined['date'].min().date()} through {combined['date'].max().date()}")
print(combined.head().to_string(index=False))

Usable joined rows: 2,741
Date range: 2015-01-05 through 2025-12-31
      date  wti_price_usd_per_barrel  daily_return_pct  temp_min_f  temp_max_f  temp_avg_f
2015-01-05                     50.05         -5.064492        14.0        43.0        28.5
2015-01-06                     47.98         -4.135864        23.0        41.0        32.0
2015-01-07                     48.69          1.479783         9.0        29.0        19.0
2015-01-08                     48.80          0.225919         7.0        37.0        22.0
2015-01-09                     48.35         -0.922131        15.0        33.0        24.0


### Likeliest failure point and fallback

The least trusted part of the plan is using one nearby station as a proxy for Cushing temperature; if Stillwater 2 W has too many missing observations or is not geographically representative enough, I will use the next-nearest NOAA station with complete temperature coverage or combine multiple nearby stations into a daily regional average.

## 3. Constraints and Guardrails

### EIA terms

**Applicable terms:** [EIA API Terms of Service Agreement](https://www.eia.gov/opendata/terms-of-service.php)

> “You may use the EIA API to develop a service to search, display, analyze, retrieve, view and otherwise ‘get’ information from EIA data.”

The terms also state:

> “You should use the ‘EIA’ or the ‘U.S. Energy Information Administration’ names in order to identify the source of API content.”

This project uses the API for analysis and identifies the source as the U.S. Energy Information Administration. It will not imply EIA endorsement or falsely present modified content as official EIA content.

### NOAA/NCEI constraints

**Applicable constraints:** [NCEI GHCN-Daily Version 3 metadata—Constraints section](https://www.ncei.noaa.gov/access/metadata/landing-page/bin/iso?id=gov.noaa.ncdc:C00861)

> “Cite as: Menne, Matthew J., Imke Durre, Bryant Korzeniewski, Shelley McNeill, Kristy Thomas, Xungang Yin, Steven Anthony, Ron Ray, Russell S. Vose, Byron E. Gleason, and Tamara G. Houston (2012): Global Historical Climatology Network - Daily (GHCN-Daily), Version 3. [indicate subset used]. NOAA National Climatic Data Center. doi:10.7289/V5D21VHZ [access date].”

The metadata also warns:

> “NOAA and NCEI cannot provide any warranty as to the accuracy, reliability, or completeness of furnished data. Users assume responsibility to determine the usability of these data.”

I will cite the GHCN-Daily dataset and document that the project uses the Stillwater 2 W subset. I will also check missing observations and will not treat the station record as error-free.

### Course guardrail certification

1. **Approved collection method:** Both sources use documented government APIs/web services. No prohibited website scraping is planned.
2. **No login or paywall:** Both requests access public data without entering a private account or bypassing a paywall.
3. **No personal or identifiable information:** The project contains only station-level weather and commodity-price observations, not information about individuals.
4. **Rate limiting:** The notebook makes one request to each source, does not paginate aggressively, and will pause or retry conservatively if a source reports a rate limit.
5. **Honest identification and caching:** The request identifies itself as an educational DSA 405 project. Retrieved results will be saved locally during later milestones so the same historical period is not downloaded repeatedly. Both publishers will be attributed accurately.

## 4. Tier Declaration

**Tier 1 — Solid.** This project combines two public sources through a one-to-one date join, and both are retrieved through documented government API/web-service endpoints; this matches the Tier 1 definition and is an appropriate scope for my current data-analysis experience.

## 5. Self-Scored P1 Rubric

| Criterion | Weight | Self-score | Evidence in this notebook |
|---|---:|---:|---|
| Question and motivation | ×1 | **3/3** | A one-sentence association question names the place, variables, and period; the motivation explains why either result matters. |
| Sources and access evidence | ×2 | **3/3** | Both sources are precisely named with actual dataset/API URLs, coverage, numeric sizes, access methods, HTTP 200 outputs, previews, and a demonstrated join. |
| Applicable constraints and guardrails | ×2 | **3/3** | Source-specific terms are linked and quoted, attribution is addressed, and all five course guardrails are certified by name. |
| Tier declaration and fit | ×1 | **3/3** | Tier 1 is declared and matches two documented web sources with a one-to-one date join. |

**Weighted self-score:** `(3×1) + (3×2) + (3×2) + (3×1) = 18/18`.

### Final checklist

- [x] One answerable sentence states the question.
- [x] Two sources are named with publisher, actual URL, coverage, time span, size, and access method.
- [x] Both sources come from documented web services.
- [x] HTTP status, row count, and preview outputs demonstrate access.
- [x] The likely failure point and fallback are stated.
- [x] The applicable constraint for each source is linked and quoted.
- [x] Every course guardrail is addressed by name.
- [x] Tier 1 is declared with a fit statement.
- [x] A self-scored rubric is included.



## AI Use:
AI was used to help reformat the jupyter notebook, to verify if I can find sources for my research question, finding the sources and verifying if I can use the provided sources. I also used AI to help prove the source is reachable via code. 